# The Riemann zeta function and the primes

*maths · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/maths/zeta_primes)

Evaluate $\zeta(s)$ on the critical line $\operatorname{Re} s = \frac12$ for thousands of heights at once, find its first nontrivial zeros as sign changes of the real function $Z(t)$, and count primes with a boolean tensor sieve. Then Riemann's explicit formula rebuilds the staircase $\pi(x)$ from the zeros: each zero adds one wave, and with more zeros the smooth curve sharpens into steps at the primes.

**Vector form**

$$\zeta(s) = \sum_{n\ge1} n^{-s} = \prod_p \frac{1}{1 - p^{-s}},\qquad Z(t) = e^{i\theta(t)}\,\zeta(\tfrac12 + it) \in \mathbb R,\qquad J(x) = \operatorname{li}(x) - \sum_\rho \operatorname{li}(x^\rho) - \ln 2 + \int_x^\infty \frac{dt}{t(t^2-1)\ln t}$$

**Index (tensor) form**

$$\zeta_t = E_{tn}1_n + \frac{N^{1-s_t}}{s_t - 1} + \frac{N^{-s_t}}{2} + \sum_{k} \frac{B_{2k}}{(2k)!}\,(s_t)_{2k-1}\,N^{-s_t-2k+1},\quad E_{tn} = n^{-s_t},\qquad \pi_K(x) = \sum_{n} \frac{\mu(n)}{n}\Big[\operatorname{li}(x^{1/n}) - \sum_{k\le K} 2\,\mathrm{Re}\operatorname{Ei}\big(\rho_k \tfrac{\ln x}{n}\big) - \ln 2 + I(x^{1/n})\Big]$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $s_t = \tfrac12 + it$ | `0.5 + 1j * t` | (T,) complex128 | many heights $t$ on the critical line at once |
| $E_{tn}1_n$ | `einsum('tn->t', exp(-s[:, None] * log(n)))` | (T, N) → (T,) | repeated $n$ ⇒ summed: the first $N-1$ terms of the Dirichlet series |
| $\frac{B_{2k}}{(2k)!}(s_t)_{2k-1}N^{-s_t-2k+1}$ | `b / factorial(2k) * rising * n_pow / big**(2k-1)` | (T,) | Euler–Maclaurin corrections with Bernoulli numbers; $(s)_{2k-1} = s(s+1)\cdots(s+2k-2)$. Valid for every $s \ne 1$ |
| $Z(t)$ | `(exp(1j * theta(t)) * zeta(0.5 + 1j * t, N)).real` | (T,) | a real function with the same zeros as $\zeta(\frac12 + it)$; $\theta(t)$ from its asymptotic series |
| $\gamma_k$ | `find_zeros(t, z, N)` | (K,) | sign changes of $Z$ on a grid with step 0.01, then 48 bisection steps on all brackets at once |
| $[n \text{ prime}]$ | `is_prime[p * p :: p] = False` | (x_max + 1,) bool | sieve of Eratosthenes: one strided slice per prime $p \le \sqrt{x}$; $\pi(x)$ is its cumulative sum |
| $\operatorname{li}(x^{\rho})$ | `ei(rho[:, None] * log(x)[None, :])` | (K, X) complex | every zero at every $x$; conjugate zeros together give $2\,\mathrm{Re}$ |
| $\sum_{k\le K}$ | `torch.cumsum(..., dim=0)` | (K, X) | the approximation for every $K$ at once: one frame per row |
| $\mu(n)$ | `mobius(64)` | list | Möbius function; the sum over $n$ stops at $x^{1/n} < 2$ because $J(y) = 0$ there |

The series $\sum n^{-s}$ converges only for $\operatorname{Re} s > 1$; Euler–Maclaurin turns the tail into an integral plus Bernoulli corrections, which continues $\zeta$ to the whole plane except the pole at $s = 1$. On the critical line $Z(t) = e^{i\theta(t)}\zeta(\frac12 + it)$ is real, so a sign change of $Z$ proves a zero lies in between. The computed zeros agree with the published values $14.134725\ldots$, $21.022040\ldots$, $25.010858\ldots$ to about $10^{-9}$, and the number found below height $T$ matches the Riemann–von Mangoldt count $N(T) \approx \theta(T)/\pi + 1$, a check that none was missed. The faster Riemann–Siegel formula (main sum plus the first correction) is shown for comparison: its error is already only $10^{-3}$ to $10^{-2}$ at the first zeros and shrinks as $t$ grows. The prime number theorem says $\pi(x) \sim x/\ln x$, but $\operatorname{li}(x)$ is much closer, and Riemann's $R(x) = \sum_n \frac{\mu(n)}{n}\operatorname{li}(x^{1/n})$ closer still (middle panel; computed with Gram's series). Riemann (1859) showed that the remaining error is carried entirely by the zeros: $J(x) = \sum_{p^m \le x} \frac1m$ equals $\operatorname{li}(x) - \sum_\rho \operatorname{li}(x^\rho) - \ln 2 + \int_x^\infty \frac{dt}{t(t^2-1)\ln t}$ (proved by von Mangoldt in 1895), and Möbius inversion gives $\pi(x)$. Each pair of zeros $\rho = \frac12 \pm i\gamma$ adds a wave of size about $\sqrt{x}/(\gamma\ln x)$ and frequency $\gamma$ in $\ln x$; truncating the sum leaves Gibbs-like ripples near each step. The Riemann hypothesis states that every nontrivial zero (every zero with $0 < \operatorname{Re} s < 1$) has real part exactly $\frac12$. It is unproved. It has been verified by computer for more than $10^{12}$ zeros, and it is equivalent to the error bound $\pi(x) = \operatorname{li}(x) + O(\sqrt{x}\ln x)$ (von Koch, 1901). This program only finds zeros on the line; it cannot rule out zeros off it. Further reading: H. M. Edwards, Riemann's Zeta Function (1974).


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/maths/zeta_primes/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors
from .equation import gram_r, li, riemann_siegel_z, theta


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    n_zeros = 20 if fast_mode() else params["n_zeros"]

    # 1. zeros of Z(t) on the critical line
    t_max = tensors.height_for(n_zeros)
    n_terms = solve.n_terms_for(t_max)
    t = tensors.t_grid(t_max, 0.01, dev)  # (T,)
    z = solve.z_on_grid(t, n_terms)
    gammas = solve.find_zeros(t, z, n_terms)
    expected = int(theta(torch.tensor(t_max, dtype=torch.float64)) / math.pi + 1)
    print(f"Z(t) on {len(t)} points of 10 ≤ t ≤ {t_max:.1f} (Euler–Maclaurin, N = {n_terms})")
    print(f"sign changes found: {len(gammas)}; zero count N(T) ≈ θ(T)/π + 1 = {expected}")
    for g, known in zip(gammas[:5].tolist(), tensors.KNOWN_ZEROS):
        print(f"  γ = {g:.9f}   known {known:.9f}   difference {abs(g - known):.1e}")
    rs = riemann_siegel_z(gammas[:5])
    print(f"Riemann–Siegel (main sum + C₀) at those zeros: |Z| ≤ {rs.abs().max():.1e} instead of 0")
    gammas = gammas[:n_zeros]

    # 2. π(x) from the sieve, compared with x/ln x, li(x) and R(x)
    x_max = 10 ** (5 if fast_mode() else params["log10_x"])
    pi = tensors.prime_pi(tensors.sieve(x_max, dev))  # (x_max + 1,)
    zeta_int = tensors.zeta_at_integers(100, dev)
    print(f"{'x':>9} {'π(x)':>8} {'x/ln x':>10} {'li(x)':>10} {'R(x)':>10}")
    for e in range(1, int(math.log10(x_max)) + 1):
        x = torch.tensor([10.0**e], dtype=torch.float64, device=dev)
        print(
            f"{10**e:>9} {int(pi[10**e]):>8} {(x / torch.log(x)).item():>10.1f} "
            f"{li(x).item():>10.1f} {gram_r(x, zeta_int).item():>10.1f}"
        )
    x_log = torch.unique(torch.logspace(1, math.log10(x_max), 400, dtype=torch.float64, device=dev).floor())
    approx = {
        "x / ln x": x_log / torch.log(x_log),
        "li(x)": li(x_log),
        "R(x)": gram_r(x_log, zeta_int),
    }

    # 3. explicit formula: π(x) rebuilt from the first K zeros, K = 0 … n_zeros
    x_lin = torch.linspace(2, params["x_explicit"], 1500, dtype=torch.float64, device=dev)
    mu = tensors.mobius(64)
    pi_k = solve.prime_count_from_zeros(x_lin, gammas, mu)  # (K + 1, X)
    exact = pi[x_lin.floor().long()].double()
    for k in (0, min(10, n_zeros), n_zeros):
        err = (pi_k[k] - exact).abs()
        print(f"explicit formula with {k:3d} zeros: mean |π_K(x) − π(x)| = {err.mean():.3f} on 2 ≤ x ≤ {x_lin[-1]:.0f}")
    return plot.animate(t, z, gammas, x_log, pi[x_log.long()], approx, x_lin, exact, pi_k, out_dir / "zeta_primes.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/maths/zeta_primes/equation.py
r"""Mathematics → tensor translation.

1. ζ(s) by Euler–Maclaurin (valid for every s ≠ 1, not only Re s > 1), vectorised over many s_t at once:

    ζ(s_t) = E_{tn} 1_n + N^{1−s_t}/(s_t − 1) + ½ N^{−s_t} + Σ_k B_{2k}/(2k)! · s_t(s_t+1)⋯(s_t+2k−2) · N^{−s_t−2k+1}

   with E_{tn} = n^{−s_t} for n = 1 … N−1 (n summed) and B_{2k} the Bernoulli numbers.

2. On the critical line s = ½ + it the Riemann–Siegel function Z(t) = e^{iθ(t)} ζ(½ + it) is real, so its zeros are
   sign changes. θ(t) = arg Γ(¼ + it/2) − (t/2) ln π has the asymptotic series used below.

3. Primes from zeros. Riemann's prime-power count J(x) = Σ_{p^m ≤ x} 1/m obeys the explicit formula

    J(x) = li(x) − Σ_ρ li(x^ρ) − ln 2 + ∫_x^∞ dt / (t (t² − 1) ln t),

   summed over the zeros ρ = ½ + iγ in conjugate pairs, li(x^ρ) := Ei(ρ ln x). Möbius inversion gives
   π(x) = Σ_n μ(n)/n · J(x^{1/n}), a finite sum because J(y) = 0 for y < 2.
"""

import math

import torch

EULER_GAMMA = 0.5772156649015329
BERNOULLI = [1 / 6, -1 / 30, 1 / 42, -1 / 30, 5 / 66, -691 / 2730, 7 / 6, -3617 / 510, 43867 / 798, -174611 / 330]


def zeta(s: torch.Tensor, n_terms: int) -> torch.Tensor:
    r"""ζ(s_t) by Euler–Maclaurin with N = n_terms and 10 Bernoulli corrections.   s: (T,) complex128 → (T,)"""
    n = torch.arange(1, n_terms, dtype=torch.float64, device=s.device)  # 1 … N−1
    head = torch.einsum("tn->t", torch.exp(-s[:, None] * torch.log(n)[None, :]))  # E_{tn} 1_n
    big = torch.tensor(float(n_terms), dtype=torch.float64, device=s.device)
    n_pow = torch.exp(-s * torch.log(big))  # N^{−s}
    total = head + big * n_pow / (s - 1) + 0.5 * n_pow
    rising = s.clone()  # s(s+1)⋯(s+2k−2)
    for k, b in enumerate(BERNOULLI, start=1):
        total = total + b / math.factorial(2 * k) * rising * n_pow / big ** (2 * k - 1)
        rising = rising * (s + 2 * k - 1) * (s + 2 * k)
    return total


def theta(t: torch.Tensor) -> torch.Tensor:
    r"""Riemann–Siegel θ(t) = t/2 ln(t/2π) − t/2 − π/8 + 1/(48t) + 7/(5760t³) + …   (accurate to < 1e-8 for t > 10)."""
    return t / 2 * torch.log(t / (2 * math.pi)) - t / 2 - math.pi / 8 + 1 / (48 * t) + 7 / (5760 * t**3)


def hardy_z(t: torch.Tensor, n_terms: int) -> torch.Tensor:
    r"""Z(t) = Re[e^{iθ(t)} ζ(½ + it)] (the imaginary part vanishes up to rounding).   (T,) → (T,)"""
    return (torch.exp(1j * theta(t)) * zeta(0.5 + 1j * t, n_terms)).real


def riemann_siegel_z(t: torch.Tensor) -> torch.Tensor:
    r"""Riemann–Siegel formula: main sum Z ≈ 2 Σ_{n≤N} n^{−½} cos(θ − t ln n) plus the first correction term
    (−1)^{N−1} (t/2π)^{−¼} C₀(p), N = ⌊√(t/2π)⌋, p = √(t/2π) − N, C₀(p) = cos 2π(p² − p − 1/16) / cos 2πp."""
    a = torch.sqrt(t / (2 * math.pi))
    big_n = a.floor()
    n = torch.arange(1, int(big_n.max()) + 1, dtype=torch.float64, device=t.device)  # (N_max,)
    keep = n[None, :] <= big_n[:, None]  # only n ≤ N(t) counts
    phase = theta(t)[:, None] - t[:, None] * torch.log(n)[None, :]  # (T, N_max)
    main = 2 * torch.einsum("tn->t", torch.where(keep, torch.cos(phase) / torch.sqrt(n), 0.0))
    p = a - big_n
    den = torch.cos(2 * math.pi * p)
    p = torch.where(den.abs() < 1e-9, p + 1e-9, p)  # C₀ has removable 0/0 points at p = ¼, ¾
    c0 = torch.cos(2 * math.pi * (p * p - p - 1 / 16)) / torch.cos(2 * math.pi * p)
    sign = 1 - 2 * ((big_n - 1) % 2)  # (−1)^{N−1}
    return main + sign * a.rsqrt() * c0  # a^{−1/2} = (t/2π)^{−1/4}


def ei(w: torch.Tensor) -> torch.Tensor:
    r"""Exponential integral Ei(w) for complex w (principal branch, cut along the negative real axis).

    |w| < 24: power series Ei(w) = γ + ln w + Σ_k w^k / (k·k!).
    |w| ≥ 24: asymptotic series Ei(w) ≈ e^w/w Σ_{k<24} k!/w^k + iπ·sign(Im w), error below ~1e-9 relative.
    """
    w = w.to(torch.complex128)
    small = w.abs() < 24
    ws = torch.where(small, w, 1.0)
    term, series = ws.clone(), ws.clone()
    for k in range(2, 100):
        term = term * ws * (k - 1) / k**2  # w^k / (k·k!)
        series = series + term
    near = EULER_GAMMA + torch.log(ws) + series
    wl = torch.where(small, 30.0, w)
    term, tail = torch.ones_like(wl), torch.ones_like(wl)
    for k in range(1, 24):
        term = term * k / wl
        tail = tail + term
    far = torch.exp(wl) / wl * tail + 1j * math.pi * torch.sign(wl.imag)
    return torch.where(small, near, far)


def li(x: torch.Tensor) -> torch.Tensor:
    r"""Logarithmic integral li(x) = ∫₀^x dt/ln t (principal value) = Ei(ln x), for real x > 1."""
    return ei(torch.log(x.to(torch.float64))).real


def tail_integral(y: torch.Tensor) -> torch.Tensor:
    r"""∫_y^∞ dt / (t (t² − 1) ln t) = ∫_{ln y}^∞ du / ((e^{2u} − 1) u), trapezoid rule; < 0.15 for y ≥ 2."""
    u = torch.log(y)[:, None] + torch.linspace(0, 12, 3000, dtype=torch.float64, device=y.device)[None, :]  # (Y, U)
    return torch.trapezoid(1 / ((torch.exp(2 * u) - 1) * u), u, dim=1)


def zero_terms(x: torch.Tensor, gammas: torch.Tensor) -> torch.Tensor:
    r"""li(x^ρ) + li(x^ρ̄) = 2 Re Ei(ρ ln x) for every zero ρ_k = ½ + iγ_k and every x.   (X,), (K,) → (K, X)"""
    rho = 0.5 + 1j * gammas.to(torch.float64)
    return 2 * ei(rho[:, None] * torch.log(x)[None, :]).real


def gram_r(x: torch.Tensor, zeta_int: torch.Tensor) -> torch.Tensor:
    r"""Riemann's R(x) = Σ_n μ(n)/n li(x^{1/n}) via the Gram series 1 + Σ_k (ln x)^k / (k·k!·ζ(k+1)).
    zeta_int[k] = ζ(k+1) for k = 1 … K.   (X,) → (X,)"""
    lx = torch.log(x.to(torch.float64))
    total, power = torch.ones_like(lx), torch.ones_like(lx)
    for k in range(1, len(zeta_int)):
        power = power * lx / k  # (ln x)^k / k!
        total = total + power / (k * zeta_int[k])
    return total


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/maths/zeta_primes/tensors.py
"""Build the concrete tensors: the prime sieve, the Möbius function, ζ at the integers and the sample grids."""

import math

import torch

from .equation import theta, zeta

# Well-established values (Odlyzko's tables) used only to check the computation.
KNOWN_ZEROS = [14.134725142, 21.022039639, 25.010857580, 30.424876126, 32.935061588]
KNOWN_PI = {10: 4, 100: 25, 1000: 168, 10**4: 1229, 10**5: 9592, 10**6: 78498, 10**7: 664579}


def sieve(x_max: int, device: torch.device) -> torch.Tensor:
    """Sieve of Eratosthenes as a boolean tensor is_prime[n] for n = 0 … x_max: (x_max + 1,)."""
    is_prime = torch.ones(x_max + 1, dtype=torch.bool, device=device)
    is_prime[:2] = False
    for p in range(2, math.isqrt(x_max) + 1):
        if is_prime[p]:
            is_prime[p * p :: p] = False  # strike every multiple from p² on in one slice
    return is_prime


def prime_pi(is_prime: torch.Tensor) -> torch.Tensor:
    """π(n) for n = 0 … x_max: the running count of primes.   (x_max + 1,) int64"""
    return torch.cumsum(is_prime.long(), dim=0)


def mobius(n_max: int) -> list[int]:
    """μ(n) for n = 0 … n_max (μ(0) unused): 0 if n has a square factor, else (−1)^(number of prime factors)."""
    mu = [1] * (n_max + 1)
    for p in range(2, n_max + 1):
        if all(p % q for q in range(2, math.isqrt(p) + 1)):  # p is prime
            for m in range(p, n_max + 1, p):
                mu[m] = -mu[m]
            for m in range(p * p, n_max + 1, p * p):
                mu[m] = 0
    return mu


def zeta_at_integers(k_max: int, device: torch.device) -> torch.Tensor:
    """ζ(k + 1) for k = 0 … k_max (index 0 unused), for Gram's series of R(x)."""
    s = torch.arange(1, k_max + 2, dtype=torch.float64, device=device).to(torch.complex128)
    s[0] = 2.0  # placeholder for the unused slot (ζ has a pole at s = 1)
    return zeta(s, 30).real


def height_for(n_zeros: int) -> float:
    """Height T with about n_zeros zeros below it, from the zero count N(T) ≈ θ(T)/π + 1 (bisection)."""
    lo, hi = 10.0, 10000.0
    for _ in range(60):
        mid = 0.5 * (lo + hi)
        lo, hi = (mid, hi) if theta(torch.tensor(mid, dtype=torch.float64)) / math.pi + 1 < n_zeros + 0.5 else (lo, mid)
    return hi + 1.0


def t_grid(t_max: float, step: float, device: torch.device) -> torch.Tensor:
    """Heights t on the critical line from 10 (below the first zero) to t_max: (T,)."""
    return torch.arange(10.0, t_max, step, dtype=torch.float64, device=device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/maths/zeta_primes/solve.py
"""Find the zeros of Z(t) and rebuild π(x) from them with the explicit formula."""

import math

import torch

from problems._common.progress import steps

from .equation import hardy_z, li, tail_integral, zero_terms


def n_terms_for(t_max: float) -> int:
    """Euler–Maclaurin needs N well above |s|/2π; N = t/2 + 20 keeps the error below 1e-12."""
    return int(t_max / 2) + 20


def z_on_grid(t: torch.Tensor, n_terms: int, chunk: int = 4096) -> torch.Tensor:
    """Z(t) for a long grid, in chunks so the (T, N) term matrix stays small."""
    return torch.cat([hardy_z(t[i : i + chunk], n_terms) for i in range(0, len(t), chunk)])


def find_zeros(t: torch.Tensor, z: torch.Tensor, n_terms: int, n_bisect: int = 48) -> torch.Tensor:
    """Sign changes of Z on the grid bracket the zeros; all brackets are bisected at once.   → (K,) heights γ_k"""
    change = (z[:-1] * z[1:]) < 0
    lo, hi = t[:-1][change].clone(), t[1:][change].clone()
    z_lo = z[:-1][change].clone()
    for _ in steps(n_bisect, "bisection"):
        mid = 0.5 * (lo + hi)
        z_mid = hardy_z(mid, n_terms)
        same = torch.sign(z_mid) == torch.sign(z_lo)
        lo, z_lo = torch.where(same, mid, lo), torch.where(same, z_mid, z_lo)
        hi = torch.where(same, hi, mid)
    return 0.5 * (lo + hi)


def prime_count_from_zeros(x: torch.Tensor, gammas: torch.Tensor, mu: list[int]) -> torch.Tensor:
    """π_K(x) = Σ_n μ(n)/n · J_K(x^{1/n}) using the first K = 0 … len(gammas) zeros, all at once.

    J_K(y) = li(y) − Σ_{k≤K} 2 Re li(y^{ρ_k}) − ln 2 + ∫_y^∞ dt/(t(t²−1) ln t), and J(y) = 0 for y < 2.
    Returns (K + 1, X): row K is the approximation with K zeros; row 0 has none (the smooth part, close to R(x)).
    """
    out = torch.zeros(len(gammas) + 1, len(x), dtype=torch.float64, device=x.device)
    for n in range(1, int(math.log2(x.max().item())) + 1):
        if mu[n] == 0:
            continue
        y = x ** (1 / n)
        smooth = li(y) - math.log(2) + tail_integral(y)  # (X,)
        waves = torch.cumsum(zero_terms(y, gammas), dim=0)  # (K, X): sum over the first k zeros
        j = torch.cat([smooth[None], smooth[None] - waves])  # (K + 1, X)
        out += mu[n] / n * torch.where(y >= 2, j, 0.0)
    return out


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/maths/zeta_primes/plot.py
"""Z(t) with its zeros, π(x) against the classical approximations, and π(x) rebuilt from more and more zeros."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif


def animate(
    t: torch.Tensor,
    z: torch.Tensor,
    gammas: torch.Tensor,
    x_log: torch.Tensor,
    pi_log: torch.Tensor,
    approx: dict,
    x_lin: torch.Tensor,
    exact: torch.Tensor,
    pi_k: torch.Tensor,
    path: Path,
) -> Path:
    t, z, g = t.cpu().numpy(), z.cpu().numpy(), gammas.cpu().numpy()
    fig, (ax_z, ax_p, ax_e) = plt.subplots(1, 3, figsize=(11, 3.9), gridspec_kw={"width_ratios": [1.1, 1, 1.3]})

    show = t <= 60
    ax_z.plot(t[show], z[show], color="#374151", lw=1)
    ax_z.axhline(0, color="#9ca3af", lw=0.8)
    g60 = g[g <= 60]
    used = ax_z.scatter(g60, np.zeros_like(g60), s=22, zorder=3, color="#9ca3af")
    ax_z.set_xlabel("$t$")
    ax_z.set_title(r"$Z(t) = e^{i\theta(t)}\zeta(1/2 + it)$", fontsize=9)

    x, p = x_log.cpu().numpy(), pi_log.cpu().numpy().astype(float)
    for (name, a), colour in zip(approx.items(), ACCENT[1:]):
        ax_p.loglog(x, np.abs(a.cpu().numpy() / p - 1), color=colour, lw=1.3, label=name)
    ax_p.set_xlabel("$x$")
    ax_p.set_title(r"relative error $|\mathrm{approx}/\pi(x) - 1|$", fontsize=9)
    ax_p.legend(fontsize=7, loc="lower left")

    xl, ex, pk = x_lin.cpu().numpy(), exact.cpu().numpy(), pi_k.cpu().numpy()
    ax_e.step(xl, ex, where="post", color="#374151", lw=1.2, label=r"$\pi(x)$ (sieve)")
    ax_e.plot(xl, pk[0], color="#9ca3af", lw=1, ls="--", label="no zeros (≈ R(x))")
    (curve,) = ax_e.plot(xl, pk[0], color="#2563eb", lw=1.3, label="with zeros")
    ax_e.set_xlabel("$x$")
    ax_e.set_ylim(-1, ex.max() * 1.1 + 1)
    ax_e.legend(fontsize=7, loc="upper left")
    title = ax_e.set_title("", fontsize=9)

    n_max = len(g)
    counts = sorted({round(n_max * (f / 59) ** 2) for f in range(60)})  # slow at first, faster later

    def update(f: int):
        k = counts[f]
        curve.set_ydata(pk[k])
        used.set_color(["#dc2626" if j < k else "#9ca3af" for j in range(len(g60))])
        title.set_text(f"explicit formula with the first {k} zeros")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(counts), len(counts)), path, fps=8)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_zeros': 100,
    'x_explicit': 60.0,
    'log10_x': 7
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.maths.zeta_primes")]:
    del sys.modules[name]
main = importlib.import_module("problems.maths.zeta_primes.main")
problem_dir = Path("problems/maths/zeta_primes")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
